### 0.测试 my Fastest_oracle

In [4]:
import os
import re
import sys
import math
import time
import tempfile
import traceback
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
import json
from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# 一级测试数据集
datasets_name = "parler_data"
# 一级数据集下根据查询和图结构的差异划分的子测试数据集
# dataset_name = "dataset_test"
dataset_name = "dataset_one"
# 原始CSV数据路径
CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
# 转换后GraphLib数据存放路径
Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"
# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.run_without_author_user_post()
# converter.remove_edge_labels()
# dataset_name = "dataset_one"
runner = FastestRunner(build_dir="/home/wangshuo/projects/FaSTest-main/build")
print(dataset_name)
sample_budget = 21000
# 默认执行 ./Fastest -d parler --ROOT_LABEL 1 (表示推理谓词所在节点的标签)
code, output = runner.run(dataset=dataset_name, root_label=-1,sample_budget=sample_budget)

dataset_one
🚀 正在运行: /home/wangshuo/projects/FaSTest-main/build/Fastest -d dataset_one --ROOT_LABEL -1 --SAMPLE_BUDGET 21000

[CLI] Root label set to -1
[CLI] Sample budget set to 21000
[info] SAMPLE_BUDGET:21000
[Info] Cleared old estimate file: /home/wangshuo/resource/datasets/parler_data/dataset_one/results/in_estimateW_result.txt
[info] args:0x7fff5eda9408
/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/parler_ans.txt
[Info] Reading core labels configuration from: /home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph/user_custom_labels.txt
[Info] Core labels configuration file not found. Running in global estimation mode only.
[infor] root label-1
Constructing Candidate Space: 12 20
[info] Query Size:16

Start Processing /home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph/query_dense_1_1.graph for global cardinality...
[Info] Auto-selected query root index 1 with (transferred) label 1
[Check] Sum(node_est) = 129165, est_total = 12

In [ ]:
%load_ext autoreload
%autoreload 2

### 1.测试三种不同实现 compute_T_true 的速度差异：pandas、polars、duckdb。我给你写一个独立的测试脚本，可以多次运行取平均耗时。

In [ ]:
import os
import time
import pandas as pd
import polars as pl
import duckdb

# ===============================================================
#  函数定义区 - 请确保您使用的是您项目中最新的函数版本
#  这些函数已经支持通过关键字参数传递 gt_match_col
# ===============================================================

def compute_T_true(gt_path, id_mapping_path, post_csv_path, **kwargs):
    try:
        gt_df = pd.read_csv(gt_path)
        if gt_df.empty: return 0.0
        idmap_df = pd.read_csv(id_mapping_path)
        post_df = pd.read_csv(post_csv_path)
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        print(f"  [Pandas] 警告: 无法读取或文件为空: {os.path.basename(gt_path)}")
        return 0.0
    post_map = idmap_df[idmap_df["type"].str.lower() == "post"][["internal_id", "orig_id"]].copy()
    gt_match_col = kwargs.get("gt_match_col", "u1") # <-- 从 kwargs 获取列名
    prob_col = kwargs.get("prob_col", "ML1_oracle1_probability")
    prob_threshold = kwargs.get("prob_threshold", 0.5)
    if gt_match_col not in gt_df.columns:
        print(f"  [Pandas] 警告: 列 '{gt_match_col}' 不存在于 {os.path.basename(gt_path)}")
        return 0.0
    gt_df = gt_df[[gt_match_col]].rename(columns={gt_match_col: "internal_id"})
    merged = gt_df.merge(post_map, on="internal_id", how="left")
    post_counts = merged.groupby("orig_id").size().reset_index(name="st_truth")
    post_df = post_df.merge(post_counts, left_on="id:ID", right_on="orig_id", how="left")
    post_df["st_truth"] = post_df["st_truth"].fillna(0)
    post_df["oracle"] = (post_df[prob_col] > prob_threshold).astype(int)
    post_df["true_contrib"] = post_df["st_truth"] * post_df["oracle"]
    T_true = post_df["true_contrib"].sum()
    print(f"  [Pandas] T_true = {T_true:.3f}")
    return T_true

def compute_T_true_polars(gt_path, id_mapping_path, post_csv_path, **kwargs):
    gt_match_col = kwargs.get("gt_match_col", "u1") # <-- 从 kwargs 获取列名
    prob_col = kwargs.get("prob_col", "ML1_oracle1_probability")
    prob_threshold = kwargs.get("prob_threshold", 0.5)
    try:
        gt_df = pl.scan_csv(gt_path)
        idmap_df = pl.scan_csv(id_mapping_path)
        post_df = pl.scan_csv(post_csv_path)
    except Exception: return 0.0
    post_map = idmap_df.filter(pl.col("type").str.to_lowercase() == "post").select(["internal_id", "orig_id"])
    gt_df = gt_df.select(pl.col(gt_match_col).alias("internal_id"))
    merged = gt_df.join(post_map, on="internal_id", how="left")
    post_counts = merged.group_by("orig_id").agg(pl.count().alias("st_truth"))
    final_lazy_df = post_df.join(post_counts, left_on="id:ID", right_on="orig_id", how="left").fill_null(0).with_columns(
        (pl.col(prob_col) > prob_threshold).cast(pl.Int8).alias("oracle"),
    ).with_columns(
        (pl.col("st_truth") * pl.col("oracle")).alias("true_contrib")
    )
    result = final_lazy_df.select(pl.sum("true_contrib").alias("T_true")).collect()
    T_true_value = result["T_true"][0] if result is not None and len(result) > 0 else 0.0
    print(f"  [Polars] T_true = {T_true_value:.3f}")
    return T_true_value

def compute_T_true_duckdb(gt_path, id_mapping_path, post_csv_path, **kwargs):
    con = duckdb.connect()
    gt_match_col = kwargs.get("gt_match_col", "u1") # <-- 从 kwargs 获取列名
    prob_col = kwargs.get("prob_col", "ML1_oracle1_probability")
    prob_threshold = kwargs.get("prob_threshold", 0.5)
    try:
        # 使用参数化查询防止SQL注入，并正确处理列名
        con.execute(f'CREATE OR REPLACE TABLE gt AS SELECT "{gt_match_col}" AS internal_id FROM read_csv_auto(?);', [gt_path])
        con.execute('CREATE OR REPLACE TABLE idmap AS SELECT internal_id, orig_id, type FROM read_csv_auto(?);', [id_mapping_path])
        con.execute(f'CREATE OR REPLACE TABLE posts AS SELECT "id:ID" AS id, "{prob_col}" AS prob FROM read_csv_auto(?);', [post_csv_path])
        gt_count = con.execute("SELECT COUNT(*) FROM gt").fetchone()[0]
        if gt_count == 0:
            con.close()
            return 0.0
        query = f"SELECT SUM(cnt * CAST((prob > {prob_threshold}) AS INTEGER)) FROM (SELECT p.id, p.prob, COALESCE(pc.cnt, 0) AS cnt FROM posts p LEFT JOIN (SELECT im.orig_id AS id, COUNT(*) AS cnt FROM gt g JOIN idmap im ON g.internal_id = im.internal_id WHERE LOWER(im.type) = 'post' GROUP BY im.orig_id) pc ON p.id = pc.id)"
        result = con.execute(query).fetchone()
        T_true = result[0] if result and result[0] is not None else 0.0
    except duckdb.Error as e:
        print(f"  [DuckDB] 警告: 执行失败 {os.path.basename(gt_path)} - {e}")
        return 0.0
    finally:
        con.close()
    print(f"  [DuckDB] T_true = {T_true:.3f}")
    return T_true

# --- 主执行区 ---

if __name__ == "__main__":
    # --- 1. 配置参数 ---
    gt_directory = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/structure_result/"
    idmap_file = "/home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph/id_mapping.csv"
    post_csv = "/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data/post.csv"
    # 新增：gt_match_col 映射文件的路径
    gt_match_col_file = "/home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph/infer_node.txt"
    
    RUN_TIMES = 2

    # --- 2. 发现并排序 Ground Truth 文件 ---
    try:
        gt_files_unsorted = [f for f in os.listdir(gt_directory) if f.endswith('.csv') and os.path.getsize(os.path.join(gt_directory, f)) > 0]
        # 关键：对文件名进行排序，以确保与 infer_node.txt 的顺序匹配
        gt_files_sorted = sorted(gt_files_unsorted)
        all_gt_files = [os.path.join(gt_directory, f) for f in gt_files_sorted]

        if not all_gt_files:
            print(f"❌ 错误: 在目录 '{gt_directory}' 中没有找到任何非空的 .csv 文件。")
            exit()
        print(f"📂 发现并排序了 {len(all_gt_files)} 个 Ground Truth 文件待处理。")
    except FileNotFoundError:
        print(f"❌ 错误: 目录不存在 '{gt_directory}'")
        exit()

    # --- 新步骤: 读取并映射 gt_match_col ---
    try:
        with open(gt_match_col_file, 'r') as f:
            gt_match_cols = [line.strip() for line in f if line.strip()]
        print(f"📄 从 '{os.path.basename(gt_match_col_file)}' 中读取了 {len(gt_match_cols)} 个列名。")
    except FileNotFoundError:
        print(f"❌ 错误: gt_match_col 文件不存在 '{gt_match_col_file}'")
        exit()

    # 验证文件和列名的数量是否匹配
    if len(all_gt_files) != len(gt_match_cols):
        print(f"❌ 严重错误: 文件数量 ({len(all_gt_files)}) 与列名数量 ({len(gt_match_cols)}) 不匹配！")
        print("请检查 'infer_node.txt' 和 ground truth 目录中的文件。")
        exit()
    
    # 创建从文件路径到其特定 gt_match_col 的映射字典
    file_to_col_map = dict(zip(all_gt_files, gt_match_cols))
    print("✅ 文件路径与列名映射成功。\n")

    # --- 3. 初始化计时器 ---
    pandas_file_avg_times, polars_file_avg_times, duckdb_file_avg_times = [], [], []
    functions_to_benchmark = {
        # "Pandas": (compute_T_true, pandas_file_avg_times),
        "Polars": (compute_T_true_polars, polars_file_avg_times),
        "DuckDB": (compute_T_true_duckdb, duckdb_file_avg_times),
    }

    # --- 4. 遍历所有文件并执行基准测试 ---
    for i, gt_file_path in enumerate(all_gt_files):
        # 从映射中获取当前文件对应的列名
        specific_gt_match_col = file_to_col_map[gt_file_path]
        
        print(f"--- ({i+1}/{len(all_gt_files)}) 正在处理: {os.path.basename(gt_file_path)} (使用列: '{specific_gt_match_col}') ---")
        
        for name, (func, time_list) in functions_to_benchmark.items():
            run_times = []
            for _ in range(RUN_TIMES):
                start = time.perf_counter()
                # 关键：将 specific_gt_match_col 作为关键字参数传递给函数
                func(gt_file_path, idmap_file, post_csv, gt_match_col=specific_gt_match_col)
                end = time.perf_counter()
                run_times.append(end - start)
            
            avg_time_for_file = sum(run_times) / len(run_times)
            time_list.append(avg_time_for_file)
            print(f"  - {name} 平均耗时: {avg_time_for_file:.4f}s")
        print("-" * 60)

    # --- 5. 计算并打印最终总结 ---
    # final_avg_pandas = sum(pandas_file_avg_times) / len(pandas_file_avg_times) if pandas_file_avg_times else 0
    final_avg_polars = sum(polars_file_avg_times) / len(polars_file_avg_times) if polars_file_avg_times else 0
    final_avg_duckdb = sum(duckdb_file_avg_times) / len(duckdb_file_avg_times) if duckdb_file_avg_times else 0

    print("\n\n======== 📊 最终基准测试总结 ======== ")
    print(f"总共处理文件数: {len(all_gt_files)}")
    print(f"每个文件重复运行次数: {RUN_TIMES}")
    print("-" * 40)
    # print(f"🐼 Pandas 平均运行时间: {final_avg_pandas:.4f}s")
    print(f"🐻‍❄️ Polars 平均运行时间: {final_avg_polars:.4f}s")
    print(f"🦆 DuckDB 平均运行时间: {final_avg_duckdb:.4f}s")
    print("========================================")

### 2. 提高长短csv文件格式转换效率

In [ ]:
import pandas as pd
import os
from typing import Dict, Tuple, List

# --- 全局常量：定义需要的列，方便复用和裁剪 ---
EXPECTED_ML1_COLS = ['ML1_oracle1_probability', 'ML1_proxy4b1_probability', 'ML1_proxy2b1_probability']
EXPECTED_ML2_COLS = ['ML2_oracle2_probability', 'ML2_proxy2d2_probability', 'ML2_proxy4d2_probability', 'ML2_proxy1_probability']

def load_and_prepare_mappings(id_mapping_path: str) -> pd.DataFrame:
    """读取id_mapping.csv并准备好用于连接的DataFrame。"""
    if not os.path.exists(id_mapping_path):
        raise FileNotFoundError(f"ID映射文件不存在: {id_mapping_path}")
        
    id_map_df = pd.read_csv(id_mapping_path, dtype={'internal_id': str, 'orig_id': str, 'type': str})
    id_map_df.rename(columns={'internal_id': 'node_id'}, inplace=True)
    
    print(f"加载了 {len(id_map_df)} 条ID映射记录。")
    return id_map_df

def load_source_csvs_optimized(post_csv_path: str, comment_csv_path: str) -> Dict[str, pd.DataFrame]:
    """
    读取原始 CSV 文件，但仅保留需要的列以减少内存占用和加速合并。
    保持 dtype=str 以确保输出格式与原版完全一致（不丢失精度或格式）。
    """
    if not os.path.exists(post_csv_path):
        raise FileNotFoundError(f"post.csv 文件不存在: {post_csv_path}")
    if not os.path.exists(comment_csv_path):
        raise FileNotFoundError(f"comment.csv 文件不存在: {comment_csv_path}")
        
    # 读取 Post 数据
    post_df = pd.read_csv(post_csv_path, dtype=str)
    if 'id:ID' in post_df.columns:
        post_df.rename(columns={'id:ID': 'orig_id'}, inplace=True)
    
    # 【优化点1】只保留 id 和 ML1 相关列
    keep_cols_post = ['orig_id'] + [c for c in EXPECTED_ML1_COLS if c in post_df.columns]
    post_df = post_df[keep_cols_post]

    # 读取 Comment 数据
    comment_df = pd.read_csv(comment_csv_path, dtype=str)
    if 'id:ID' in comment_df.columns:
        comment_df.rename(columns={'id:ID': 'orig_id'}, inplace=True)
    
    # 【优化点1】只保留 id 和 ML2 相关列
    keep_cols_comment = ['orig_id'] + [c for c in EXPECTED_ML2_COLS if c in comment_df.columns]
    comment_df = comment_df[keep_cols_comment]
        
    print(f"加载并裁剪数据: Post {len(post_df)} 行, Comment {len(comment_df)} 行。")
    return {"Post": post_df, "Comment": comment_df}

def precompute_node_lookups(id_map_df: pd.DataFrame, sources: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """
    【优化点2】在循环外预先将 id_mapping 和 源数据合并。
    创建从 node_id 直接到 (orig_id, ML_values) 的映射表。
    """
    print("正在预计算节点属性查找表...")
    
    # 1. 准备 Post 查找表
    # 筛选出 id_map 中类型为 Post 的节点
    map_posts = id_map_df[id_map_df['type'] == 'Post'][['node_id', 'orig_id']]
    # Left join 保证即使源数据缺失，节点也在（这与原逻辑一致）
    post_lookup = pd.merge(map_posts, sources['Post'], on='orig_id', how='left')
    
    # 2. 准备 Comment 查找表
    map_comments = id_map_df[id_map_df['type'] == 'Comment'][['node_id', 'orig_id']]
    comment_lookup = pd.merge(map_comments, sources['Comment'], on='orig_id', how='left')
    
    print(f"预计算完成。Post查找表: {len(post_lookup)} 行, Comment查找表: {len(comment_lookup)} 行。")
    return {"Post": post_lookup, "Comment": comment_lookup}

def process_single_query_file_optimized(query_file_path: str, lookups: Dict[str, pd.DataFrame], output_dir: str):
    """
    处理单个查询文件。
    使用预计算的 lookups 替代原始的多次 merge。
    """
    query_basename = os.path.basename(query_file_path).replace("_estimateW_result.csv", "")
    # print(f"正在处理: {query_basename}") # 减少打印频率，或者保留
    
    instance_df = pd.read_csv(query_file_path)
    if instance_df.empty:
        print(f"文件 {query_basename} 为空，跳过。")
        return
        
    # 确保连接键类型一致
    instance_df['node_id'] = instance_df['node_id'].astype(str)
    
    # 提取只需要的列进行 merge，减少开销
    nodes_in_query = instance_df[['instance_id', 'node_id']]

    # 1. Post 数据流处理
    # 使用 inner join：因为 lookups['Post'] 只包含 Post 类型的 node_id。
    # 如果 instance_df 中的某个 node_id 是 Comment，它在这里自然会被过滤掉，不需要显式的 type 判断。
    posts_joined = pd.merge(nodes_in_query, lookups['Post'], on='node_id', how='inner')
    
    # 2. Comment 数据流处理
    comments_joined = pd.merge(nodes_in_query, lookups['Comment'], on='node_id', how='inner')
    
    # 3. 聚合逻辑 (保持原有逻辑不变)
    
    # --- 聚合 Post ---
    actual_ml1_cols = [col for col in EXPECTED_ML1_COLS if col in posts_joined.columns]
    
    if not posts_joined.empty:
        agg_dict = {col: list for col in actual_ml1_cols}
        agg_dict['orig_id'] = list
        
        agg_posts = posts_joined.groupby('instance_id').agg(agg_dict).reset_index()
        agg_posts.rename(columns={'orig_id': 'post_id_list'}, inplace=True)
    else:
        agg_posts = pd.DataFrame(columns=['instance_id', 'post_id_list'] + actual_ml1_cols)

    # --- 聚合 Comment ---
    actual_ml2_cols = [col for col in EXPECTED_ML2_COLS if col in comments_joined.columns]
    
    if not comments_joined.empty:
        agg_dict = {col: list for col in actual_ml2_cols}
        agg_dict['orig_id'] = list
        
        agg_comments = comments_joined.groupby('instance_id').agg(agg_dict).reset_index()
        agg_comments.rename(columns={'orig_id': 'comment_id_list'}, inplace=True)
    else:
        agg_comments = pd.DataFrame(columns=['instance_id', 'comment_id_list'] + actual_ml2_cols)
        
    # 4. 最终合并
    # 仅从原始 df 取出所需的聚合基准列，去重
    base_agg_df = instance_df[['instance_id', 'estimateW', 'global_estimateW']].drop_duplicates(subset=['instance_id'])
    
    final_df = pd.merge(base_agg_df, agg_posts, on='instance_id', how='left')
    final_df = pd.merge(final_df, agg_comments, on='instance_id', how='left')

    # 5. 保存结果
    output_filename = f"aggregated_list_{query_basename}.csv"
    output_filepath = os.path.join(output_dir, output_filename)
    
    # 修正：final_columns_order 引用了局部变量 expected_ml...，这里使用全局变量
    final_columns_order = [
        'instance_id', 'estimateW', 'global_estimateW', 
        'post_id_list', 'comment_id_list'
    ] + EXPECTED_ML1_COLS + EXPECTED_ML2_COLS
    
    final_df = final_df.reindex(columns=final_columns_order)
    final_df.to_csv(output_filepath, index=False)
    # print(f"已保存: {output_filepath}")

def main(dataset_name):
    """主执行函数"""
    print(f"====== 开始处理数据集: {dataset_name} (优化版) ======")
    
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    estimate_dir = os.path.join(base_path, "results", "structure_estimate")
    id_mapping_path = os.path.join(base_path, "data_graph", "id_mapping.csv")
    post_csv_path = os.path.join(base_path, "csv_data", "post.csv")
    comment_csv_path = os.path.join(base_path, "csv_data", "comment.csv")
    output_dir = os.path.join(base_path, "results", "aggregated_results2")
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"创建输出目录: {output_dir}")

    try:
        # 1. 加载映射
        id_map_df = load_and_prepare_mappings(id_mapping_path)
        
        # 2. 加载并裁剪源数据 (优化点)
        sources = load_source_csvs_optimized(post_csv_path, comment_csv_path)
        
        # 3. 预计算查找表 (优化点：核心提速)
        # 将 id_mapping 和 sources 提前 merge 好
        node_lookups = precompute_node_lookups(id_map_df, sources)

        query_files = [f for f in os.listdir(estimate_dir) if f.endswith('.csv')]
        if not query_files:
            print(f"[警告] 无文件需处理: {estimate_dir}")
            return
        
        print(f"开始处理 {len(query_files)} 个查询文件...")
        
        # 计数器用于进度显示
        count = 0
        total = len(query_files)
        
        for query_file in sorted(query_files):
            query_file_path = os.path.join(estimate_dir, query_file)
            
            # 传入预计算好的 lookups
            process_single_query_file_optimized(query_file_path, node_lookups, output_dir)
            
            count += 1
            if count % 10 == 0:
                print(f"进度: {count}/{total}")
            
        print(f"\n====== 数据集 {dataset_name} 处理完毕 ======")

    except FileNotFoundError as e:
        print(f"[严重错误] 依赖文件未找到: {e}")
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f"[严重错误] 处理过程中发生异常: {e}")

if __name__ == '__main__':
    dataset_name_to_process = 'dataset_test'
    main(dataset_name_to_process)

### 3.测试原有的函数parse_sv_multi 和parse_sv_file 对科学计数法的支持情况

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
%load_ext autoreload
%autoreload 2
"""
测试 FastestEstimateMerger 的 parse_sv_file() 与 parse_sv_multi() 
在科学计数法输入下的解析能力。
"""

import os
import pandas as pd
from pythonProject.src.Structure_first.fastest_pipeline import FastestEstimateMerger

# ======== Step 1. 构造临时测试数据 ========
os.makedirs("./tmp_test", exist_ok=True)

# --- 单查询文件 (parse_sv_file 测试用) ---
sv_single_path = "./tmp_test/sv_estimate_result.txt"
with open(sv_single_path, "w") as f:
    f.write("""Query: query_dense_1_1.graph
All Est: 3.4567e+06
51095 1.10788e+06
51096 5.43210e+03
51097 7.8
""")

# --- 多查询文件 (parse_sv_multi 测试用) ---
sv_multi_path = "./tmp_test/in_estimateW_result.txt"
with open(sv_multi_path, "w") as f:
    f.write("""Query: query_dense_1_1.graph
All Est: 1.23e+05
51095 1.10788e+06
51096 9.87e+03
Query: query_path_4_0.graph
All Est: 2.5e+04
100 2.1e+01
101 5.0e+00
""")

# ======== Step 2. 创建类实例 ========
dataset_name = 'dataset_two'
SV_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/in_estimateW_result.txt"
INFER_NODE_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/infer_node.txt"
IDMAP_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/id_mapping.csv"
POST_CSV = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/csv_data/post.csv"
GT_RESULT_DIR = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth/structure_result"
OUTPUT_DIR = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results"
POST_WITH_ESTIMATE_CSV = os.path.join(OUTPUT_DIR, "post_with_estimate.csv")
SUMMARY_CSV = os.path.join(OUTPUT_DIR, "results_summary.csv")
SUMMARY_TXT = os.path.join(OUTPUT_DIR, "results_summary.txt")

merger = FastestEstimateMerger(
    sv_file=sv_single_path,
    map_file=IDMAP_FILE,
    post_file=POST_CSV
)

# ======== Step 3. 测试 parse_sv_file() ========
print("\n===== 🧪 测试 parse_sv_file() =====")
merger.parse_sv_file()
print(merger.sv_df)
print("\n估计值类型:", merger.sv_df["estimate"].apply(type).unique())
print("是否全部为浮点数:", merger.sv_df["estimate"].apply(lambda x: isinstance(x, float)).all())

